# CICIoT2023 — Exploratory Data Analysis

**ML Concepts covered:**
- Class imbalance visualisation
- Feature distribution analysis (violin plots, box plots)
- Correlation heatmap — detecting redundant features
- PCA — visualising high-dimensional data in 2D
- t-SNE — non-linear dimensionality reduction for cluster inspection

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('tab10')
%matplotlib inline

In [ ]:
# Load processed training split
PROC = Path('../data/processed')
train = pd.read_parquet(PROC / 'train.parquet')
val   = pd.read_parquet(PROC / 'val.parquet')
test  = pd.read_parquet(PROC / 'test.parquet')

df = pd.concat([train, val, test], ignore_index=True)
print(f'Total samples: {len(df):,}')
print(f'Features: {df.shape[1]-1}')
df.head()

## 1. Class Distribution

Understanding class imbalance is the first step in IDS evaluation.

In [ ]:
counts = df['label'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts.plot(kind='bar', ax=axes[0], color=sns.color_palette('tab10'))
axes[0].set_title('Class Frequency', fontsize=13)
axes[0].set_xlabel('Attack Family')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

counts.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Class Distribution', fontsize=13)
axes[1].set_ylabel('')

plt.suptitle('CICIoT2023 — Class Balance', fontsize=14)
plt.tight_layout()
plt.savefig('../reports/eda_class_distribution.png', dpi=150)
plt.show()
print('\nImbalance ratio:', round(counts.max()/counts.min(), 1), 'x')

## 2. Feature Distributions

Violin plots show the full distribution shape — useful for detecting heavy tails and bimodal distributions that simple box plots miss.

In [ ]:
# Select a subset of informative features
key_features = ['Rate', 'Srate', 'Drate', 'syn_flag_number', 'ack_flag_number',
                 'flow_duration', 'Tot sum', 'Std', 'IAT', 'Number']
available = [f for f in key_features if f in df.columns]

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()

for i, feat in enumerate(available[:10]):
    data = []
    labels_list = []
    for cls in df['label'].unique():
        vals = df.loc[df['label']==cls, feat].dropna().clip(
            upper=df[feat].quantile(0.99))
        data.append(vals.values)
        labels_list.append(cls)
    axes[i].violinplot(data, showmedians=True)
    axes[i].set_title(feat, fontsize=10)
    axes[i].set_xticks(range(1, len(labels_list)+1))
    axes[i].set_xticklabels(labels_list, rotation=45, fontsize=7)

plt.suptitle('Feature Distributions per Class', fontsize=14)
plt.tight_layout()
plt.savefig('../reports/eda_feature_distributions.png', dpi=150)
plt.show()

## 3. Correlation Matrix

High correlation between features means they carry redundant information.  
For Logistic Regression: correlated features cause multicollinearity → unstable coefficients.  
For tree models: less of an issue, but still increases computational cost.

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns.tolist()
corr = df[numeric_cols[:20]].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0,
            annot=False, ax=ax, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Matrix (lower triangle)', fontsize=14)
plt.tight_layout()
plt.savefig('../reports/eda_correlation.png', dpi=150)
plt.show()

## 4. PCA — Linear Dimensionality Reduction

PCA finds orthogonal axes of maximum variance.  
If classes are linearly separable, a good Logistic Regression baseline is expected.  
If classes overlap heavily in PCA space, we need non-linear models (trees, MLP).

In [ ]:
sample = df.sample(min(10000, len(df)), random_state=42)
X_s = sample[numeric_cols].fillna(sample[numeric_cols].median()).values
X_s = StandardScaler().fit_transform(X_s)

pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_s)

fig, ax = plt.subplots(figsize=(10, 7))
classes = sample['label'].unique()
for cls in classes:
    mask = sample['label'].values == cls
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], s=5, alpha=0.4, label=cls)

ax.set_title(f'PCA — 2D projection (explained variance: {pca.explained_variance_ratio_.sum():.1%})', fontsize=13)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.legend(markerscale=3, fontsize=9)
plt.tight_layout()
plt.savefig('../reports/eda_pca.png', dpi=150)
plt.show()

## 5. t-SNE — Non-linear Dimensionality Reduction

t-SNE preserves local neighbourhood structure better than PCA for complex, non-linear manifolds.  
Well-separated clusters → model should achieve high per-class recall.  
Overlapping clusters → expect confusion between those classes.

In [ ]:
# t-SNE is slow — use a small sample
tsne_sample = df.sample(min(3000, len(df)), random_state=42)
X_tsne_in = tsne_sample[numeric_cols].fillna(tsne_sample[numeric_cols].median()).values
X_tsne_in = StandardScaler().fit_transform(X_tsne_in)

tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne = tsne.fit_transform(X_tsne_in)

fig, ax = plt.subplots(figsize=(10, 7))
for cls in tsne_sample['label'].unique():
    mask = tsne_sample['label'].values == cls
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1], s=8, alpha=0.5, label=cls)
ax.set_title('t-SNE — 2D Embedding of Traffic Features', fontsize=13)
ax.legend(markerscale=2, fontsize=9)
plt.tight_layout()
plt.savefig('../reports/eda_tsne.png', dpi=150)
plt.show()

## 6. Missing Values & Data Quality

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if len(missing) > 0:
    missing.plot(kind='bar', figsize=(10, 4))
    plt.title('Missing Values per Feature')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.savefig('../reports/eda_missing.png', dpi=150)
    plt.show()
else:
    print('No missing values in this subset.')

print('\nData types:')
print(df.dtypes.value_counts())
print(f'\nDuplicate rows: {df.duplicated().sum():,}')